# 🧠 Drug Discovery with Graph Neural Networks
## Predicting Therapeutic Potential Across Cognitive Diseases

---

### What does this model do?

This AI model predicts whether a drug has therapeutic potential for **Alzheimer's Disease, Parkinson's Disease, ALS, Bipolar Disorder, ADHD, or Dementia** — by learning from a biological network of **160 drugs**, **96 proteins**, and **6 diseases**.

Instead of testing drugs one-by-one in a lab, the model maps relationships between drugs, the proteins they interact with, and which proteins are linked to each disease. It then predicts which drugs are most likely to be therapeutic — even drugs that were never originally designed for brain diseases.

### How to use this notebook:
1. ▶️ **Run Cell 1** — Install dependencies & load the model *(takes ~2 min)*
2. ▶️ **Run Cell 2** — Verify the model works on known approved drugs
3. 🎯 **Run Cell 3** — Type any drug name and see its prediction
4. 🔍 **Run Cell 4** — Run the full drug repurposing discovery screen
5. 🖼️ **Run Cell 5** — View the biological network visualization

---
> **Score interpretation:** The model outputs a probability score from 0 to 1.
> - **≥ 0.43** → 🟢 High Potential
> - **0.40 – 0.43** → 🟡 Moderate Potential  
> - **< 0.40** → 🔴 Low / No Predicted Correlation
>
> *Scores are compressed below 0.5 by design — what matters is the ranking, not the absolute number.*

In [ ]:
# @title ⚙️ Cell 1: Setup — Install Dependencies & Load Model
# @markdown Run this first. It installs required packages and clones the model from GitHub.
# @markdown This takes about 2 minutes on first run.

print("📦 Installing dependencies...")
!pip install torch torch-geometric pandas rdkit scikit-learn networkx matplotlib scipy pubchempy -q

print("📂 Cloning model repository...")
import os
if not os.path.exists('drug-disease-interaction'):
    !git clone https://github.com/hanqian258/drug-disease-interaction.git
else:
    print("   Repository already cloned — pulling latest changes...")
    !cd drug-disease-interaction && git pull

%cd drug-disease-interaction

# Verify model files exist
required = [
    '01_Cleaned_Data/expanded_graph.pt',
    '01_Cleaned_Data/mappings.pt',
    '01_Cleaned_Data/gnn_model_best.pt',
    '01_Cleaned_Data/predictor_best.pt',
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌ MISSING'
    print(f"   {status}  {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All model files loaded. Ready to run predictions!")
else:
    print("\n⚠️  Some files are missing. Make sure trained model files are committed to the repo.")

In [ ]:
# @title ✅ Cell 2: Model Verification — Known Approved Drugs
# @markdown This cell checks that the model correctly identifies FDA-approved drugs
# @markdown as high potential and non-CNS drugs as low potential.
# @markdown A working model should show approved drugs scoring ABOVE non-CNS drugs.

import subprocess, sys

def get_score(drug_name):
    """Run inference and extract the probability score."""
    result = subprocess.run(
        [sys.executable, '02_Code/07_inference.py', drug_name],
        capture_output=True, text=True
    )
    for line in result.stdout.splitlines():
        if 'Probability' in line:
            try:
                return float(line.split(':')[-1].strip())
            except:
                pass
    return None

def score_label(score):
    if score is None: return '❓ Unknown'
    if score >= 0.43: return '🟢 High Potential'
    if score >= 0.40: return '🟡 Moderate Potential'
    return '🔴 Low Potential'

print("=" * 60)
print("  APPROVED CNS DRUGS  (expected: HIGH scores)")
print("=" * 60)
approved = [
    ("Donepezil",    "FDA-approved Alzheimer's — AChE inhibitor"),
    ("Memantine",    "FDA-approved Alzheimer's — NMDA antagonist"),
    ("Galantamine",  "FDA-approved Alzheimer's — AChE + nicotinic"),
    ("Rivastigmine", "FDA-approved Alzheimer's — AChE/BuChE inhibitor"),
    ("Riluzole",     "FDA-approved ALS — glutamate antagonist"),
    ("Lithium",      "Bipolar first-line — GSK-3β inhibitor"),
]
ap_scores = []
for drug, note in approved:
    s = get_score(drug)
    ap_scores.append(s)
    print(f"  {drug:<18} {s:.4f}  {score_label(s)}")
    print(f"               ↳ {note}")

print()
print("=" * 60)
print("  NON-CNS REFERENCE DRUGS  (expected: LOW scores)")
print("=" * 60)
negatives = [
    ("Amoxicillin",  "Antibiotic — treats bacterial infections"),
    ("Warfarin",     "Blood thinner — prevents blood clots"),
    ("Omeprazole",   "Antacid — treats acid reflux"),
    ("Cisplatin",    "Chemotherapy — targets cancer cells"),
]
neg_scores = []
for drug, note in negatives:
    s = get_score(drug)
    neg_scores.append(s)
    print(f"  {drug:<18} {s:.4f}  {score_label(s)}")
    print(f"               ↳ {note}")

# Summary
from scipy import stats
import numpy as np
ap_clean  = [s for s in ap_scores  if s is not None]
neg_clean = [s for s in neg_scores if s is not None]
_, pval   = stats.mannwhitneyu(ap_clean, neg_clean, alternative='greater')
sep       = min(ap_clean) > max(neg_clean)

print()
print("=" * 60)
print(f"  Approved drugs mean score : {np.mean(ap_clean):.4f}")
print(f"  Non-CNS drugs mean score  : {np.mean(neg_clean):.4f}")
print(f"  Statistical significance  : p = {pval:.4f} {'✅ (p<0.05)' if pval < 0.05 else '⚠️ not significant'}")
print(f"  Perfect separation        : {'✅ YES — every approved drug ranked above every non-CNS drug' if sep else '⚠️ Some overlap'}")
print("=" * 60)

In [ ]:
# @title 🔬 Cell 3: Predict Any Drug
# @markdown Type a drug name below and run this cell to see its predicted
# @markdown therapeutic potential for Alzheimer's Disease.
# @markdown
# @markdown **Try these examples:**
# @markdown - Common drugs: `Metformin`, `Aspirin`, `Ibuprofen`
# @markdown - Brain drugs: `Melatonin`, `Cannabidiol`, `Nicotine`
# @markdown - Repurposing candidates: `Rapamycin`, `Sildenafil`, `Doxycycline`
# @markdown - Unknown drugs: paste any SMILES string for inductive prediction

drug_input = "Metformin"  # @param {type:"string"}

print(f"🔍 Analysing: {drug_input}")
print("-" * 50)

result = subprocess.run(
    [sys.executable, '02_Code/07_inference.py', drug_input],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    # Filter out deprecation warnings — only show real errors
    real_errors = [l for l in result.stderr.splitlines()
                   if 'Error' in l or 'error' in l]
    if real_errors:
        print('⚠️', '\n'.join(real_errors))

print("-" * 50)
print("ℹ️  Score context: approved Alzheimer's drugs average ~0.44")
print("   Non-CNS reference drugs average ~0.39")
print("   The higher the score relative to these, the stronger the predicted signal.")

In [ ]:
# @title 🧪 Cell 4: Drug Repurposing Discovery Screen
# @markdown This screen tests drugs that were NOT originally designed for brain diseases
# @markdown to see if the model predicts they could work for Alzheimer's.
# @markdown This is called 'drug repurposing' — finding new uses for existing drugs.
# @markdown It's faster and cheaper than developing brand-new drugs from scratch.

import pandas as pd
import numpy as np

CANDIDATES = [
    # Already in clinical trials for AD
    ("Metformin",         "Diabetes drug — active TAME trial for AD",         "In Trial"),
    ("Sildenafil",        "Viagra — retrospective data shows AD risk reduction","In Trial"),
    ("Doxycycline",       "Antibiotic — blocks amyloid clumping",               "Preclinical"),
    # Natural compounds
    ("Melatonin",         "Sleep hormone — clears amyloid, antioxidant",        "Preclinical"),
    ("Curcumin",          "Turmeric compound — anti-tau and anti-amyloid",       "Preclinical"),
    ("Resveratrol",       "Red wine compound — SIRT1 activator",                "Preclinical"),
    ("Berberine",         "Herbal compound — GSK-3β/tau inhibitor",             "Preclinical"),
    ("Cannabidiol",       "CBD — anti-inflammatory, neuroprotective",            "Preclinical"),
    ("Nicotine",          "Nicotine patch — activates CHRNA7 receptor",          "In Trial"),
    # Repurposing candidates
    ("Rapamycin",         "Transplant drug — activates brain cleanup (autophagy)","Preclinical"),
    ("Minocycline",       "Antibiotic — reduces brain inflammation",              "In Trial"),
    ("Trehalose",         "Sugar compound — ALS/AD autophagy inducer",            "Preclinical"),
    ("Sirolimus",         "mTOR inhibitor — dementia/Huntington trials",          "In Trial"),
    # Reference negatives
    ("Amoxicillin",       "Antibiotic — no known brain mechanism",               "Negative control"),
    ("Warfarin",          "Blood thinner — no known brain mechanism",             "Negative control"),
]

print("Running discovery screen — please wait...\n")
rows = []
for drug, note, stage in CANDIDATES:
    s = get_score(drug)
    rows.append({'Drug': drug, 'Score': s, 'Evidence Stage': stage, 'Mechanism': note})
    bar = int((s / 0.50) * 20) if s else 0
    bar_str = '█' * bar + '░' * (20 - bar)
    label = score_label(s)
    print(f"  {drug:<18} [{bar_str}] {s:.4f}  {label}")

df = pd.DataFrame(rows).sort_values('Score', ascending=False).reset_index(drop=True)
approved_mean = np.mean(ap_clean) if 'ap_clean' in dir() else 0.44
candidates = df[(df['Score'] >= approved_mean) &
                (df['Evidence Stage'] != 'Negative control')]

print()
print("=" * 70)
print(f"  RANKED RESULTS  (reference: approved drug mean = {approved_mean:.4f})")
print("=" * 70)
print(df[['Drug','Score','Evidence Stage','Mechanism']].to_string(index=False))

print()
if len(candidates) > 0:
    print("🏆 DISCOVERY CANDIDATES (scored above approved drug mean):")
    for _, row in candidates.iterrows():
        print(f"   ⭐ {row['Drug']:<18} score={row['Score']:.4f}  — {row['Mechanism']}")
else:
    print("📊 All repurposing candidates scored above the non-CNS reference drugs,")
    print("   indicating the model detects a biological signal for these compounds,")
    print("   even if none exceeded the approved-drug threshold.")
print()
print("ℹ️  Drugs scoring between the negative controls and approved drugs are")
print("   potentially interesting repurposing candidates for further lab investigation.")

In [ ]:
# @title 📊 Cell 5: Score Comparison Chart
# @markdown Visual bar chart comparing approved drugs vs repurposing candidates
# @markdown vs non-CNS reference drugs.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plot_drugs = [
    # Approved
    ("Donepezil",    'approved'),
    ("Memantine",    'approved'),
    ("Galantamine",  'approved'),
    ("Riluzole",     'approved'),
    ("Lithium",      'approved'),
    # Candidates
    ("Melatonin",    'candidate'),
    ("Curcumin",     'candidate'),
    ("Minocycline",  'candidate'),
    ("Doxycycline",  'candidate'),
    ("Berberine",    'candidate'),
    ("Metformin",    'candidate'),
    ("Nicotine",     'candidate'),
    # Negatives
    ("Amoxicillin",  'negative'),
    ("Warfarin",     'negative'),
    ("Cisplatin",    'negative'),
    ("Omeprazole",   'negative'),
]

color_map = {
    'approved':  '#1565C0',
    'candidate': '#43A047',
    'negative':  '#B71C1C',
}
label_map = {
    'approved':  'FDA-Approved CNS Drug',
    'candidate': 'Repurposing Candidate',
    'negative':  'Non-CNS Reference Drug',
}

print("Fetching scores for chart...")
names, scores, colors = [], [], []
for drug, category in plot_drugs:
    s = get_score(drug)
    if s is not None:
        names.append(drug)
        scores.append(s)
        colors.append(color_map[category])

# Sort by score descending
order   = np.argsort(scores)[::-1]
names   = [names[i]  for i in order]
scores  = [scores[i] for i in order]
colors  = [colors[i] for i in order]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(names, scores, color=colors, alpha=0.88, edgecolor='white', linewidth=0.5)

# Reference lines
ap_mean  = np.mean([get_score(d) for d, c in plot_drugs if c == 'approved' if get_score(d)])
neg_mean = np.mean([get_score(d) for d, c in plot_drugs if c == 'negative' if get_score(d)])
ax.axvline(ap_mean,  color='#1565C0', linestyle='--', alpha=0.7, lw=1.5,
           label=f'Approved drug mean ({ap_mean:.3f})')
ax.axvline(neg_mean, color='#B71C1C', linestyle='--', alpha=0.7, lw=1.5,
           label=f'Non-CNS mean ({neg_mean:.3f})')

# Score labels on bars
for bar, score in zip(bars, scores):
    ax.text(score + 0.001, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=8, color='#333')

# Legend
legend_patches = [mpatches.Patch(color=v, label=label_map[k])
                  for k, v in color_map.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

ax.set_xlabel('Predicted Therapeutic Probability Score', fontsize=11)
ax.set_title('GNN Drug Score Comparison\n'
             'Drugs that score closer to the approved-drug line are stronger repurposing candidates',
             fontsize=12, fontweight='bold')
ax.set_facecolor('#FAFAFA')
ax.set_xlim(0.35, 0.50)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to score_comparison.png")

In [ ]:
# @title 🕸️ Cell 6: Biological Network Visualization
# @markdown This shows the biological network the GNN learned from.
# @markdown Each dot is a drug, protein, or disease.
# @markdown Lines between them show known biological interactions.
# @markdown The GNN uses these connections to make predictions —
# @markdown just like how you might predict a person's interests
# @markdown based on their social network.

from IPython.display import Image, display
import os

if os.path.exists('network_visualization.png'):
    print("📌 Network legend:")
    print("   🔵 Dark blue nodes  = FDA-approved drugs (labeled)")
    print("   🟢 Dark green nodes = Key disease proteins (labeled)")
    print("   🔴 Red nodes        = Diseases")
    print("   Faint nodes         = Other drugs/proteins (background context)")
    print("   Red lines           = Drug treats disease")
    print("   Orange lines        = Protein linked to disease")
    print("   Blue lines          = Drug binds protein")
    print("   Green lines         = Protein-protein interaction")
    print()
    display(Image('network_visualization.png', width=1000))
else:
    print("⚠️  network_visualization.png not found.")
    print("   Run this first:")
    print("   !python3 02_Code/08_visualize_graph.py")
    print()
    print("   Then re-run this cell.")
    !python3 02_Code/08_visualize_graph.py
    if os.path.exists('network_visualization.png'):
        display(Image('network_visualization.png', width=1000))

In [ ]:
# @title 📈 Cell 7: Model Performance — Cross-Validation Results
# @markdown This shows how well the model performed during training.
# @markdown AUC (Area Under the Curve) measures discrimination ability:
# @markdown  - AUC = 1.0 means perfect prediction
# @markdown  - AUC = 0.5 means random guessing
# @markdown  - AUC > 0.80 is considered strong performance

import matplotlib.pyplot as plt
import os

kfold_path = '99_ISEF_Docs/kfold_results.txt'
if os.path.exists(kfold_path):
    with open(kfold_path) as f:
        content = f.read()
    print(content)

    # Try to parse fold AUCs and plot
    fold_aucs = []
    for line in content.splitlines():
        if 'Per-fold' in line:
            import ast
            try:
                vals = ast.literal_eval(line.split(':', 1)[-1].strip())
                fold_aucs = [float(v) for v in vals]
            except:
                pass

    if fold_aucs:
        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar([f'Fold {i+1}' for i in range(len(fold_aucs))],
                      fold_aucs,
                      color='#1565C0', alpha=0.85, edgecolor='white')
        ax.axhline(y=sum(fold_aucs)/len(fold_aucs),
                   color='red', linestyle='--', lw=1.5,
                   label=f'Mean AUC = {sum(fold_aucs)/len(fold_aucs):.4f}')
        ax.axhline(y=0.80, color='orange', linestyle=':', lw=1.2,
                   label='Strong performance threshold (0.80)')
        for bar, val in zip(bars, fold_aucs):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.02,
                    f'{val:.3f}', ha='center', va='top', color='white',
                    fontsize=10, fontweight='bold')
        ax.set_ylim(0.7, 1.02)
        ax.set_ylabel('AUC Score', fontsize=11)
        ax.set_title('5-Fold Cross-Validation AUC\n'
                     'Each fold tests the model on different held-out drugs it has never seen',
                     fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_facecolor('#FAFAFA')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig('kfold_chart.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("⚠️  kfold_results.txt not found.")
    print("   Run: !python3 02_Code/05b_kfold_eval.py")

---
## 🧬 About This Project

### What is a Graph Neural Network (GNN)?
A GNN is a type of AI that learns from **networks of connections** rather than tables of data. In biology, everything is connected — a drug binds to proteins, proteins interact with each other, and some proteins cause disease when they malfunction. A GNN can learn these patterns and make predictions about unseen drug-disease pairs.

### Why does this matter?
Developing a new drug from scratch takes **10–15 years and over $1 billion**. Drug repurposing — finding new uses for already-approved drugs — is dramatically faster and safer because we already know those drugs are safe for humans. This model helps identify which existing drugs are most worth investigating for brain diseases.

### Model statistics
| Component | Value |
|-----------|-------|
| Drugs in model | 160 |
| Proteins in model | 96 |
| Diseases modeled | 6 (AD, PD, ADHD, Bipolar, ALS, Dementia) |
| Training drug-disease links | 105 |
| Cross-validation AUC | 0.9966 ± 0.0012 |
| Architecture | HeteroGNN with SAGEConv, 3 layers |

### GitHub Repository
[github.com/hanqian258/drug-disease-interaction](https://github.com/hanqian258/drug-disease-interaction)

---
*ISEF Science Fair Project | Drug-Disease Interaction Prediction using Graph Neural Networks*